# 10 — System 2 Attention Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## What is System 2 Attention?

In [3]:
# System 2 Attention (S2A): Weston & Suber 2023
# Step 1: Ask model to rewrite question removing irrelevant/biasing info
# Step 2: Answer the rewritten question

original_q = (
    "My grandfather was a brilliant man who worked in finance for 40 years. "
    "He always said the stock market was unpredictable. I have been saving money for 5 years. "
    "Given how volatile the market has been and considering my grandfather's wisdom, "
    "should I invest in stocks or keep money in a savings account?"
)
print("Original question:")
print(original_q)


Original question:
My grandfather was a brilliant man who worked in finance for 40 years. He always said the stock market was unpredictable. I have been saving money for 5 years. Given how volatile the market has been and considering my grandfather's wisdom, should I invest in stocks or keep money in a savings account?


In [4]:
step1_prompt = (
    "Below is a question with potentially irrelevant context or emotional framing.\n"
    "Rewrite the question to include ONLY the information necessary to answer it well.\n"
    "Remove personal anecdotes, emotional appeals, and irrelevant background.\n\n"
    f"Original: {original_q}\n\n"
    "Rewritten question (just the essential facts and question):"
)
rewritten_q = chat([{"role":"user","content":step1_prompt}], temperature=0, max_tokens=100)
print("Step 1 - Rewritten question:")
print(rewritten_q.strip())


Step 1 - Rewritten question:
I have been saving money for 5 years. What is a suitable investment option for me, considering the volatility of the stock market?


In [5]:
step2_prompt = f"Answer this question with balanced, factual advice: {rewritten_q}"
answer = chat([{"role":"user","content":step2_prompt}], temperature=0, max_tokens=200)
print("Step 2 - Answer to rewritten question:")
print(answer.strip())

print("\n" + "="*50)
direct = chat([{"role":"user","content":original_q}], temperature=0, max_tokens=200)
print("Direct answer to original (may include bias):")
print(direct.strip())


Step 2 - Answer to rewritten question:
Congratulations on saving money for 5 years. Considering the volatility of the stock market, it's essential to choose an investment option that balances risk and potential returns. Here are some suitable options for you to consider:

1. **High-Yield Savings Account**: A high-yield savings account is a low-risk option that provides easy access to your money. It typically earns a higher interest rate than a traditional savings account, but the returns may not keep pace with inflation.
2. **Certificates of Deposit (CDs)**: CDs are time deposits offered by banks with a fixed interest rate and maturity date. They tend to be low-risk and provide a fixed return, but you'll face penalties for early withdrawal.
3. **Bonds**: Government and corporate bonds offer a relatively stable return, but their value may fluctuate with interest rates. You can consider investing in a bond fund or individual bonds with a longer maturity period to reduce volatility.
4. **

Direct answer to original (may include bias):
Your grandfather's wisdom is valuable, and his experience in finance is a great asset to consider. The stock market's unpredictability is a well-known fact, and it's essential to approach investing with caution.

Given the current market volatility, it's natural to be hesitant about investing in stocks. However, it's also essential to consider the potential long-term benefits of investing in the stock market.

Here are some points to consider:

Pros of investing in stocks:

1. **Potential for higher returns**: Historically, the stock market has provided higher returns over the long term compared to savings accounts.
2. **Inflation protection**: Stocks can help protect your purchasing power against inflation, as the value of your investments can increase over time.
3. **Diversification**: Investing in a mix of stocks can help spread risk and increase potential returns.

Cons of investing in stocks:

1. **Market volatility**: The stock market

## S2A for Opinion-Laden Questions

In [6]:
biased_q = (
    "Everyone knows Python is much better than JavaScript. "
    "I have tried JavaScript briefly and found it confusing. "
    "Don\'t you think Python is the superior language for all programming tasks?"
)

neutralize = f"Rewrite this question neutrally, removing bias and loaded assumptions:\n\n{biased_q}\n\nNeutral version:"
neutral_q = chat([{"role":"user","content":neutralize}], temperature=0, max_tokens=80)
print("Neutral question:", neutral_q.strip())

answer = chat([{"role":"user","content":f"Answer objectively: {neutral_q}"}],
              temperature=0, max_tokens=200)
print("\nBalanced answer:")
print(answer.strip())


Neutral question: What are the key differences between Python and JavaScript, and how do their respective strengths and weaknesses impact their suitability for various programming tasks?



Balanced answer:
**Overview**

Python and JavaScript are two popular programming languages with distinct differences in their design, syntax, and use cases. Understanding these differences is essential for choosing the right language for a specific task.

**Key Differences**

1. **Syntax**: Python's syntax is more concise and readable, with a focus on whitespace and indentation. JavaScript's syntax is more verbose, with a focus on semicolons and curly braces.
2. **Type System**: Python is dynamically typed, while JavaScript is also dynamically typed. However, Python has a more explicit type system, making it easier to catch type-related errors.
3. **Object-Oriented Programming (OOP)**: Both languages support OOP concepts, but Python's implementation is more explicit and flexible.
4. **Memory Management**: Python uses automatic memory management through its garbage collector, while JavaScript uses a combination of automatic and manual memory management.
5. **Platform**: Python is prima

## S2A for Reasoning Tasks with Distractors

In [7]:
distracted = (
    "John is tall, dark-haired, and works as a chef. "
    "His sister Mary is a doctor who loves hiking. "
    "Their neighbor Bob has a dog named Spot. "
    "John baked 12 cookies and ate 3. Mary took half of the remaining cookies. "
    "How many cookies does John have left?"
)

extract = f"Extract ONLY the facts needed to answer the math question. Remove all irrelevant details.\n\n{distracted}\n\nRelevant facts and question:"
core = chat([{"role":"user","content":extract}], temperature=0, max_tokens=80)
print("Core problem:", core.strip())

solve = chat([{"role":"user","content":f"Solve step by step: {core}"}],
             temperature=0, max_tokens=150)
print("\nSolution:")
print(solve.strip())


Core problem: John baked 12 cookies and ate 3. 
There are 12 - 3 = 9 cookies left. 
Mary took half of the remaining cookies. 
Half of 9 is 4.5.



Solution:
To find out how many cookies Mary took, we need to round down 4.5 to 4, since you can't take a fraction of a cookie. 

However, the correct way to find half of 9 is to divide 9 by 2, which is 4.5. But since we can't take a fraction of a cookie, we can round down to 4 or round up to 5. 

Let's assume Mary took 4 cookies. 

So, Mary took 4 cookies, and there are 9 - 4 = 5 cookies left.
